# Building AI Agents with Langchain and LLMs

This guide will walk you through the process of building AI agents using Langchain and Large Language Models (LLMs). We will cover the necessary steps to set up your environment, create an agent, and deploy it for use in later stages.

### 0. Prerequisites

- An Azure account with sufficient permissions to create resources.
- Terraform installed on your local machine.
- Azure CLI installed and configured.
- [Optional] Docker installed for building container images.

### 1. Deploy the LLM in Azure Foundry using Terraform

The terraform code for deploying the LLM and Cosmos DB is located in the `terraform` directory. You can deploy the resources by running the following commands from within the `infra` directory:

In [ ]:
! terraform -chdir=infra init -upgrade
! terraform -chdir=infra plan -out tfplan
! terraform -chdir=infra apply tfplan

Initializing the backend...
Initializing provider plugins...
- terraform.io/builtin/terraform is built in to Terraform
- Finding azure/azapi versions matching ">= 2.8.0"...
- Finding hashicorp/azurerm versions matching ">= 4.58.0"...
- Using previously-installed azure/azapi v2.10.0
- Using previously-installed hashicorp/azurerm v4.76.0

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.
terraform_data.add_serverless_gpu_profile_GPU-NC24-A100: Refreshing state... [id=e6d5aa93-b0fd-7c91-2507-da494907a7b7]
terraform_data.add_serverless_gpu_profile_GPU-NC8as-T4: Refreshing state... [id=aeb5c92a-2400-5d70-7c17-05

### 3. Get Endpoint and Key for the LLM

Retrieve the FQDN of the LLM and the API key from Terraform outputs.

In [ ]:
foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform -chdir=infra output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name)

LLM Endpoint: ╷
│ Error: Output "aca_gemma4_31b_it_a100_fqdn" not found
│ 
│ The output variable requested could not be found in the state file. If you
│ recently added this to your configuration, be sure to run `terraform
│ apply`, since the state won't be updated with new output variables until
│ that command is run.
╵
LLM Endpoint: qwen-3-6-35b-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4


### 3. Install required Python packages

In [1]:
%pip install langchain langgraph langchain-openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Set Up the LLM Model

Create a `ChatOpenAI` model pointing at the vLLM-compatible endpoint running on Azure Container Apps.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    max_completion_tokens=512
)

## 4. Test the Model

Invoke the model with a test prompt to ensure it's working correctly.

In [9]:
from langchain_core.messages import HumanMessage

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)



I'm Qwen, a large language model developed by Alibaba Group's Tongyi Lab. I'm designed to be a clear, thoughtful, and capable thinking partner that can help you with reasoning, language understanding, coding, multi-step problem solving, and adapting to different tasks or communication styles. My goal is to provide accurate, practical, and well-

## 5. Use the Model in a LangChain Agent

In [11]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[],
    checkpointer=None,
)

response = agent.stream({"messages": HumanMessage(content="Tell me about yourself")})

async for step in agent.astream(
    {"messages": [HumanMessage(content="Tell me about yourself")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about yourself
================================== Ai Message ==================================



I’m Qwen, a large language model developed by Alibaba Group’s Tongyi Lab. I’m designed to be a clear, honest, and practical thinking partner—helping with writing, coding, analysis, problem-solving, research, and more. I support many languages, work with long documents and complex files, and can adapt to different workflows and formats. 

What are you working on? I’d be glad to help
